# Notebook 07: Spatial Validation Methodology & Raster Audit

## Research Reference
**"An approach for accurate identification and monitoring of species in mangrove forests based on multi-source spectral data and deep learning"**  
*Monterrubio-Martínez, Trujillo-Acatitla, Tuxpan-Vargas, Moreno-Casasola*  
*Ecological Informatics*, Volume 85, 2025, Article 102961. DOI: [10.1016/j.ecoinf.2024.102961](https://doi.org/10.1016/j.ecoinf.2024.102961)

> ### [!IMPORTANT]
> **SCIENTIFIC AUDIT & REPRODUCIBILITY STATUS:**  
> **`DATA-LIMITED / NOT REPRODUCIBLE AT RASTER LEVEL`**
>
> **Strict Reproduction Policy:**
> - The paper evaluated trained models spatially by projecting predictions onto full-scene Sentinel-2 GeoTIFF rasters and comparing against a 3.5 cm UAV orthophoto and 100 field GPS control points in La Mancha and Arroyo Moreno (Figs. 7–10).
> - However, the authors **did not publicly release the satellite raster GeoTIFF files, the drone orthophoto, or the digitized QGIS polygon shapefiles** in their repository.
> - Under our strict scientific guidelines: **We do not download unrelated satellite scenes or fabricate missing spatial ground truth**.
> - Instead, this notebook performs a rigorous spatial data audit, examines the physical and ecological mechanisms discovered by the authors, and documents why spatial pixel inference cannot be reproduced from tabular reflectance vectors alone.

In [1]:
import sys
import os
import json
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import RAW_DATA_DIR, RESULTS_DIR

print("Auditing spatial assets in repository...")

Auditing spatial assets in repository...


## 1. Local Spatial Assets Inventory

We scan the project directory for geospatial formats (`.tif`, `.tiff`, `.shp`, `.geojson`, `.jp2`, `.gpkg`).

In [2]:
spatial_extensions = [".tif", ".tiff", ".jp2", ".shp", ".geojson", ".gpkg", ".kml"]
found_spatial_files = []

for root, _, files in os.walk(project_root):
    for f in files:
        if any(f.lower().endswith(ext) for ext in spatial_extensions):
            found_spatial_files.append(os.path.join(root, f))

print(f"Total spatial raster / vector files found: {len(found_spatial_files)}")
if len(found_spatial_files) == 0:
    print("CONFIRMED: Zero geospatial raster or vector files exist in the repository.")
    print("The dataset deposited by the authors consists solely of the tabular reflectance CSV and EML XML.")

Total spatial raster / vector files found: 0
CONFIRMED: Zero geospatial raster or vector files exist in the repository.
The dataset deposited by the authors consists solely of the tabular reflectance CSV and EML XML.


## 2. Spatial Data Availability Audit Matrix

We formalize the availability of all spatial assets required to reproduce Figures 7, 8, 9, and 10:

In [3]:
spatial_audit = [
    {
        "Asset_Name": "Monospecific Mangrove Reflectance CSV",
        "Role_in_Paper": "Training tabular data for MLP models",
        "Format": "CSV (1,605,681 rows, 10 bands)",
        "Released": True,
        "Local_Status": "Available (data/raw/DataBase_Sentinel_2_Mangrove_LaMancha.csv)"
    },
    {
        "Asset_Name": "EML Dataset Metadata",
        "Role_in_Paper": "Dataset provenance & variable definitions",
        "Format": "XML (EML v2.2.0)",
        "Released": True,
        "Local_Status": "Available (data/raw/Sentinel_2_satellite_reflectance_of_monospecific.xml)"
    },
    {
        "Asset_Name": "Sentinel-2 La Mancha Full Scene GeoTIFF",
        "Role_in_Paper": "Spatial raster for mapping binary & multiclass distributions (Figs 7-9)",
        "Format": "GeoTIFF / JP2 raster",
        "Released": False,
        "Local_Status": "UNRELEASED / NOT AVAILABLE"
    },
    {
        "Asset_Name": "UAV Drone Orthophoto (3.5 cm resolution)",
        "Role_in_Paper": "Ground truth visual baseline from 5,435 drone shots (Fig 3, Figs 7-9)",
        "Format": "High-resolution GeoTIFF",
        "Released": False,
        "Local_Status": "UNRELEASED / NOT AVAILABLE"
    },
    {
        "Asset_Name": "QGIS Species Ground Truth Polygons",
        "Role_in_Paper": "Digitized boundaries of monospecific stands & mixed mangroves",
        "Format": "Shapefile / GeoPackage",
        "Released": False,
        "Local_Status": "UNRELEASED / NOT AVAILABLE"
    },
    {
        "Asset_Name": "Arroyo Moreno Sentinel-2 Scene & Maps",
        "Role_in_Paper": "Spatial generalization site evaluation (Fig 10)",
        "Format": "GeoTIFF raster & field points",
        "Released": False,
        "Local_Status": "UNRELEASED / NOT AVAILABLE"
    },
]

df_spatial_audit = pd.DataFrame(spatial_audit)
display(df_spatial_audit)

# Export to JSON
with open(RESULTS_DIR / "spatial_audit_summary.json", "w") as f:
    json.dump(spatial_audit, f, indent=2)
print("Saved spatial audit matrix to results/spatial_audit_summary.json")

,Asset_Name,Role_in_Paper,Format,Released,Local_Status
0,Monospecific Mangrove Reflectance CSV,Training tabular data for MLP models,"CSV (1,605,681 rows, 10 bands)",True,Available (data/raw/DataBase_Sentinel_2_Mangro...
1,EML Dataset Metadata,Dataset provenance & variable definitions,XML (EML v2.2.0),True,Available (data/raw/Sentinel_2_satellite_refle...
2,Sentinel-2 La Mancha Full Scene GeoTIFF,Spatial raster for mapping binary & multiclass...,GeoTIFF / JP2 raster,False,UNRELEASED / NOT AVAILABLE
3,UAV Drone Orthophoto (3.5 cm resolution),"Ground truth visual baseline from 5,435 drone ...",High-resolution GeoTIFF,False,UNRELEASED / NOT AVAILABLE
4,QGIS Species Ground Truth Polygons,Digitized boundaries of monospecific stands & ...,Shapefile / GeoPackage,False,UNRELEASED / NOT AVAILABLE
5,Arroyo Moreno Sentinel-2 Scene & Maps,Spatial generalization site evaluation (Fig 10),GeoTIFF raster & field points,False,UNRELEASED / NOT AVAILABLE


Saved spatial audit matrix to results/spatial_audit_summary.json


## 3. Deep Analysis of the Paper's Spatial Findings (Section 3.3)

Even though the raw GeoTIFF rasters are unreleased, the authors' spatial evaluation in Section 3.3 revealed critical scientific insights regarding deep learning for remote sensing:

### A. Binary Spatial Distribution (Figure 7)
- **Accuracy vs Spatial Coherence**: While all models achieved $>99%$ test accuracy, their spatial maps differed significantly.
- **Logistic Baseline**: Displayed prominent noise and false positives in water channels.
- **Optimal Model**: The **4-hidden-layer model with 300 neurons** cleanly delineated mangrove boundaries and suppressed water confusion, making it the most reliable binary spatial model.

### B. Multiclass Spatial Distribution (Figures 8 & 9)
- **The Overfitting Paradox**: The model with the highest test accuracy (3 layers, 500 neurons: 0.9654) produced degraded spatial maps with severe water vs *Rhizophora mangle* confusion.
- **White Mangrove Overestimation**: Simpler models heavily overestimated *Laguncularia racemosa* (white mangrove).
- **The Optimal Ecological Model**: The **5-hidden-layer model with 50 neurons** was selected as the **most reliable real-world model** (Fig. 9E). Its lower neuron count per layer acted as an architectural regularizer, preventing memorization of spectral noise and yielding the cleanest spatial delineation.

### C. Arroyo Moreno Generalization (Figure 10)
- Testing on the independent Arroyo Moreno site demonstrated that models trained in La Mancha can generalize to new geographical regions without retraining, supporting broader conservation monitoring.

In [4]:
print("Spatial audit and methodology review complete.")
print("Status: Classified as DATA-LIMITED / NOT-REPRODUCIBLE AT RASTER LEVEL.")

Spatial audit and methodology review complete.
Status: Classified as DATA-LIMITED / NOT-REPRODUCIBLE AT RASTER LEVEL.
